In [ ]:
!pip install -q kaggle transformers timm scikit-learn


In [ ]:
# Upload Kaggle API key
from google.colab import files
files.upload()  # Upload kaggle.json


In [ ]:
# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset
!kaggle datasets download -d lukechugh/best-alzheimer-mri-dataset-99-accuracy
!unzip -q best-alzheimer-mri-dataset-99-accuracy.zip -d data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
from torch import nn, optim
import timm
from tqdm import tqdm
import seaborn as sns
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
DATA_DIR = 'data/Combined Dataset'
classes = sorted(os.listdir(os.path.join(DATA_DIR, 'train')))
print("Classes:", classes)

In [ ]:
# Image transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [ ]:
full_train = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_size = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(full_train, [train_size, val_size])
val_dataset.dataset.transform = val_transform

test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), transform=val_transform)


In [ ]:
# Compute class weights
labels = [label for _, label in train_dataset]
class_weights = compute_class_weight(class_weight='balanced', classes=np.arange(len(classes)), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

In [ ]:
import pandas as pd
from collections import Counter

label_counts = Counter([label for _, label in train_dataset])
df_label_dist = pd.DataFrame.from_dict(label_counts, orient='index', columns=['count'])
df_label_dist.index = [classes[i] for i in df_label_dist.index]

sns.barplot(x=df_label_dist.index, y='count', data=df_label_dist)
plt.title('Training Set Class Distribution')
plt.ylabel('Number of Samples')
plt.xlabel('Class')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


In [ ]:
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(classes))
model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
early_stop_patience = 5
def train_model(model, epochs=20):
    best_val_acc = 0
    patience_counter = 0
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct = 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == labels).sum().item()

        train_acc = train_correct / len(train_loader.dataset)
        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                val_correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = val_correct / len(val_loader.dataset)
        val_loss /= len(val_loader)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_vit_model.pth")
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print("Early stopping triggered.")
                break

    return history


In [ ]:
history = train_model(model, epochs=20)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_acc"], label="Train Acc")
plt.plot(history["val_acc"], label="Val Acc")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.legend()
plt.show()

In [ ]:
model.load_state_dict(torch.load("best_vit_model.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

# ===================== CONFUSION MATRIX =====================
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
plt.plot(history["train_loss"], label='Train Loss') # Use history["train_loss"] instead of train_losses
plt.plot(history["val_loss"], label='Val Loss')     # Use history["val_loss"] instead of val_losses
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.show()

plt.plot(history["train_acc"], label='Train Accuracy') # Use history["train_acc"] instead of train_accuracies
plt.plot(history["val_acc"], label='Val Accuracy')     # Use history["val_acc"] instead of val_accuracies
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()

In [ ]:
def show_samples(dataset, class_names):
    fig, axs = plt.subplots(len(class_names), 5, figsize=(12, 8))
    for i, cls in enumerate(class_names):
        cls_idx = classes.index(cls)
        cls_samples = [img for img, label in dataset if label == cls_idx][:5]
        for j, img in enumerate(cls_samples):
            axs[i, j].imshow(img.permute(1, 2, 0) * 0.5 + 0.5)
            axs[i, j].axis('off')
            if j == 0:
                axs[i, j].set_title(cls)
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, classes)
